# *SENTIMENT ANALYSIS FOR INDONESIAN POLTICIAL NEWS*
# this notebook using IndoBERT model

# this noteboook focused on classification of sentiment analysis (pos, neg, neutral) using transformer model of finetuned IndoBERT

# import library & define env & device

In [ ]:
!pip install -q -U "transformers<5.0.0" "accelerate>=0.26.0" "datasets" "packaging<25.0,>=23.2"

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from collections import Counter
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    precision_recall_fscore_support,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import MinMaxScaler

from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed
)
from transformers.modeling_outputs import SequenceClassifierOutput

import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Set Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Imports complete. Running on: {device}")

# Configuration & Hyperparameter defining

In [ ]:
# --- Configuration ---
SEED = 42
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 512

# Paths
DATA_PATH = "/kaggle/input/datasets/davinraffilio9/datalabeled/data_labeled"
MODEL_PATH = "/kaggle/working"


set_seed(SEED)

print(f"Model: IndoBERT")
print(f"Max Length: {MAX_LENGTH}")

# data loading & integration

In [ ]:
# Load Data
df1 = pd.read_csv("/kaggle/input/datasets/davinraffilio9/datalabeled/data_labeled/detik_labeled.csv")
df2 = pd.read_csv("/kaggle/input/datasets/davinraffilio9/datalabeled/data_labeled/cnbc_labeled.csv")
df3 = pd.read_csv("/kaggle/input/datasets/davinraffilio9/datalabeled/data_labeled/kompas_labeled.csv")

print(f"Detik: {len(df1)} | CNBC: {len(df2)} | Kompas: {len(df3)}")

# Combine Datasets
required_cols = ["date", "title", "content", "article_id", "text", "label"]

def _select_cols(df: pd.DataFrame) -> pd.DataFrame:
    return df[required_cols].copy()

df_seed = pd.concat(
    [_select_cols(df1), _select_cols(df2), _select_cols(df3)],
    ignore_index=True
)

print(f"Total Combined Samples: {len(df_seed)}")

# EDA & data preprocessing

In [ ]:

# --- 0. Setup Visualization Function ---
def plot_label_distribution(df, target_col='label', title_suffix=''):
    """
    Creates a bar plot and pie chart for label distribution.
    Auto-detects if labels are original (-1,0,1) or mapped (0,1,2).
    """
    viz_df = df.copy()
    
    # Auto-detect label format
    unique_labels = set(viz_df[target_col].unique())
    if unique_labels.issubset({0, 1, 2}):
        label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'} # Mapped ID
    else:
        label_map = {-1: 'Negative', 0: 'Neutral', 1: 'Positive'} # Original Label
        
    viz_df['label_name'] = viz_df[target_col].map(label_map)
    order = ['Negative', 'Neutral', 'Positive']
    
    # Setup plot grid
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Bar Chart (Frequency)
    sns.countplot(x='label_name', data=viz_df, order=order, palette='viridis', ax=ax1)
    ax1.set_title(f'Label Counts {title_suffix}', fontsize=12, fontweight='bold')
    ax1.set_xlabel('')
    ax1.set_ylabel('Count')
    
    # Add numbers on top of bars
    for p in ax1.patches:
        ax1.annotate(f'{int(p.get_height())}', 
                     (p.get_x() + p.get_width() / 2., p.get_height()), 
                     ha='center', va='bottom', fontsize=10, color='black')

    # 2. Pie Chart (Proportion)
    counts = viz_df['label_name'].value_counts().reindex(order)
    ax2.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140, 
            colors=sns.color_palette('viridis', 3), explode=(0.05, 0.05, 0.05))
    ax2.set_title(f'Label Proportions {title_suffix}', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# --- 1. Check & Visualize Distribution (Original Data) ---
print("="*50)
print("DATASET DISTRIBUTION (ORIGINAL)")
print("="*50)

# Text Output
print("Label Counts:")
print(df_seed['label'].map({-1:'Negative', 0:'Neutral', 1:'Positive'}).value_counts())

# Diagram Output
plot_label_distribution(df_seed, target_col='label', title_suffix='(Original df_seed)')


# --- 2. Train/Val Split ---
print("\n" + "="*50)
print("SPLITTING DATASET")
print("="*50)

train_df, val_df = train_test_split(
    df_seed,
    test_size=0.2,
    stratify=df_seed['label'],
    random_state=SEED
)

# Visualize Train Split to confirm Stratification
plot_label_distribution(train_df, target_col='label', title_suffix='(Training Split)')


# --- 3. Remap Labels (-1, 0, 1 -> 0, 1, 2) ---
label_to_id = {-1: 0, 0: 1, 1: 2}
id_to_label = {0: "negative", 1: "neutral", 2: "positive"}

train_df['label_id'] = train_df['label'].map(label_to_id)
val_df['label_id'] = val_df['label'].map(label_to_id)


# --- 4. Compute Class Weights ---
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_df['label_id']),
    y=train_df['label_id']
)

# Amplify minority class weights (Custom Adjustment)
# Logic: Give more focus to Negative/Positive, less to Neutral
class_weights = class_weights * np.array([1.5, 1.0, 1.5])
class_weights_tensor = torch.FloatTensor(class_weights).to(device)

print("\n" + "="*50)
print("TRAINING CONFIGURATION")
print("="*50)
print(f"Train Size : {len(train_df)} samples")
print(f"Val Size   : {len(val_df)} samples")
print("-" * 30)
print("Computed Class Weights (for Loss Function):")
for i, label in enumerate(['Negative', 'Neutral', 'Positive']):
    print(f"  {label:<10}: {class_weights[i]:.4f}")

# Tokenization & dataset prep

In [ ]:
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False, # Padding handled by collator later
            return_tensors=None
        )
        
        return {
            'input_ids': encoding['input_ids'],
            'attention_mask': encoding['attention_mask'],
            'labels': label
        }

# Create Dataset Objects
train_dataset = SentimentDataset(train_df['text'].tolist(), train_df['label_id'].tolist(), tokenizer, MAX_LENGTH)
val_dataset = SentimentDataset(val_df['text'].tolist(), val_df['label_id'].tolist(), tokenizer, MAX_LENGTH)

print("✅ Dataset and Tokenizer ready.")

# Model architecture definition

In [ ]:
class IndoBERTClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout_rate=0.1):
        super(IndoBERTClassifier, self).__init__()
        # Memuat model dasar IndoBERT
        self.bert = AutoModel.from_pretrained(model_name)
        
        # Layer Dropout untuk regularisasi
        self.dropout = nn.Dropout(dropout_rate)
        
        # Layer Linear langsung dari hidden size BERT (768) ke jumlah label (3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        
        # Mengambil pooled_output (representasi token [CLS])
        # Pada BERT, ini adalah representasi vektor keseluruhan kalimat
        pooled_output = outputs.pooler_output
        
        # Dropout dan Klasifikasi
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            # Menggunakan Class Weights yang sudah dihitung sebelumnya di notebook Anda
            loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
            loss = loss_fct(logits.view(-1, self.classifier.out_features), labels.view(-1))

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

# Customize trainer & metrics

In [ ]:
class FocalLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, focal_gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal_gamma = focal_gamma
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        ce_loss = F.cross_entropy(logits, labels, weight=self.class_weights, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.focal_gamma * ce_loss).mean()
        
        return (focal_loss, outputs) if return_outputs else focal_loss

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    
    f1_per_class = f1_score(labels, preds, average=None)
    
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
        'f1_negative': f1_per_class[0],
        'f1_neutral': f1_per_class[1],
        'f1_positive': f1_per_class[2],
    }

# Training initialization

In [ ]:
# Inisialisasi model murni IndoBERT
sentiment_model = IndoBERTClassifier(
    model_name=MODEL_NAME, 
    num_labels=3, 
    dropout_rate=0.1
).to(device)

print(f"Model: Murni {MODEL_NAME}")

# 2. Training Arguments
training_args = TrainingArguments(
    output_dir=f"{MODEL_PATH}/sentiment_training",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    learning_rate=3e-5,
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_dir=f"{MODEL_PATH}/logs",
    logging_steps=50,
    report_to="none",
    fp16=True,
    dataloader_num_workers=2,
    seed=SEED,
)

# 3. Initialize Trainer
trainer = FocalLossTrainer(
    class_weights=class_weights_tensor,
    focal_gamma=2.0,
    model=sentiment_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("✅ Trainer Initialized.")

# training data

In [ ]:
print("="*40)
print("🚀 STARTING TRAINING")
print("="*40)

trainer.train()

print("✅ Training Complete.")

# Final evaluation

In [ ]:
print("Running Final Evaluation...")

# Predict
pred = trainer.predict(val_dataset)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

# Metrics
final_metrics = trainer.evaluate()

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)
print(f"Accuracy    : {final_metrics['eval_accuracy']:.4f}")
print(f"Macro F1    : {final_metrics['eval_macro_f1']:.4f}")
print(f"Weighted F1 : {final_metrics['eval_weighted_f1']:.4f}")
print("-" * 50)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"], digits=4))

# analysis evaluation

In [ ]:
from wordcloud import WordCloud

# Pastikan style plot rapi
sns.set(style="whitegrid")

def plot_evaluation_results(y_true, y_pred, df_val):
    """
    Menampilkan Confusion Matrix, Grafik Metrik, dan Word Cloud.
    """
    target_names = ["Negative", "Neutral", "Positive"]
    
    # --- 1. Confusion Matrix ---
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=target_names, yticklabels=target_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.show()

    # --- 2. Per-Class Metrics Visualization (Precision, Recall, F1) ---
    from sklearn.metrics import precision_recall_fscore_support
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None)
    
    metrics_df = pd.DataFrame({
        'Class': target_names * 3,
        'Score': np.concatenate([precision, recall, f1]),
        'Metric': ['Precision']*3 + ['Recall']*3 + ['F1-Score']*3
    })
    
    plt.figure(figsize=(10, 5))
    sns.barplot(x='Class', y='Score', hue='Metric', data=metrics_df, palette='viridis')
    plt.title('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
    plt.ylim(0, 1.1)
    plt.legend(loc='lower right')
    plt.show()

    # --- 3. Word Clouds per Sentiment ---
    print("\n" + "="*50)
    print("MOST FREQUENT WORDS BY SENTIMENT")
    print("="*50)
    
    # Mapping label ID ke nama untuk filtering
    # Pastikan 'val_df' memiliki kolom 'text' dan 'label_id'
    # Kita menggunakan data validasi (df_val)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    for i, label_name in enumerate(target_names):
        # Ambil teks dari data validasi yang label aslinya = i
        # Menggunakan kolom 'text' dari val_df
        subset_text = " ".join(df_val[df_val['label_id'] == i]['text'].astype(str).tolist())
        
        if not subset_text:
            subset_text = "No Data" # Handle jika kosong
            
        wc = WordCloud(width=800, height=400, background_color='white', 
                       colormap='viridis', max_words=100).generate(subset_text)
        
        axes[i].imshow(wc, interpolation='bilinear')
        axes[i].axis('off')
        axes[i].set_title(f'{label_name} Sentiment', fontsize=16, fontweight='bold')
        
    plt.tight_layout()
    plt.show()

# --- EKSEKUSI VISUALISASI ---
# Pastikan val_df tersedia dari cell sebelumnya
plot_evaluation_results(y_true, y_pred, val_df)

# Ablation Study 1 — Dropout Rate Variants (R10 - Reviewer Response)
**Reviewer Comment 10:** Justify key hyperparameter choices.

Comparing dropout=0.1 (paper setting), 0.2, and 0.3 on the IndoBERT classification head.

**Paper Section:** Section III.C — add paragraph explaining dropout sensitivity analysis.

**Reference:** Srivastava et al. (2014). *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*. JMLR.

In [ ]:
# Ablation 1: Dropout Rate Variants
ablation1_results = []
dropout_variants = [0.1, 0.2, 0.3]

for dr in dropout_variants:
    print(f'\n--- Training with dropout={dr} ---')

    abl_model = IndoBERTClassifier(
        model_name=MODEL_NAME,
        num_labels=3,
        dropout_rate=dr
    ).to(device)

    abl_args = TrainingArguments(
        output_dir=f"{MODEL_PATH}/abl1_dr{str(dr).replace('.','')}",
        eval_strategy='epoch',
        save_strategy='no',
        learning_rate=3e-5,
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        logging_steps=200,
        report_to='none',
        fp16=True,
        seed=SEED,
    )

    abl_trainer = FocalLossTrainer(
        class_weights=class_weights_tensor,
        focal_gamma=2.0,
        model=abl_model,
        args=abl_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )
    abl_trainer.train()

    abl_pred   = abl_trainer.predict(val_dataset)
    abl_labels = abl_pred.label_ids
    abl_preds  = np.argmax(abl_pred.predictions, axis=1)
    f1_per     = f1_score(abl_labels, abl_preds, average=None)

    ablation1_results.append({
        'Dropout': dr,
        'Accuracy': accuracy_score(abl_labels, abl_preds),
        'Macro F1': f1_score(abl_labels, abl_preds, average='macro'),
        'F1-Neg':   f1_per[0],
        'F1-Pos':   f1_per[2],
        'Paper Setting': '*' if dr == 0.1 else '',
    })
    del abl_model; torch.cuda.empty_cache()

abl1_df = pd.DataFrame(ablation1_results)
print('\n=== Ablation 1: Dropout Rate Summary ===')
print(abl1_df.to_string(index=False))


# Ablation Study 2 — Learning Rate Variants (R10 - Reviewer Response)
**Reviewer Comment 10:** Justify fine-tuning learning rate selection.

Comparing lr=1e-5, 2e-5, 3e-5 (paper setting), 5e-5 following BERT fine-tuning best practices.

**Paper Section:** Section III.C — add table of LR sensitivity results.

**Reference:** Sun et al. (2019). *How to Fine-Tune BERT for Text Classification*. CCF NLPCC.

In [ ]:
# Ablation 2: Learning Rate Variants
ablation2_results = []
lr_variants = [1e-5, 2e-5, 3e-5, 5e-5]

for lr in lr_variants:
    print(f'\n--- Training with lr={lr} ---')

    abl_model = IndoBERTClassifier(
        model_name=MODEL_NAME,
        num_labels=3,
        dropout_rate=0.1
    ).to(device)

    abl_args = TrainingArguments(
        output_dir=f"{MODEL_PATH}/abl2_lr{str(lr).replace('-','').replace('.','')}",
        eval_strategy='epoch',
        save_strategy='no',
        learning_rate=lr,
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        logging_steps=200,
        report_to='none',
        fp16=True,
        seed=SEED,
    )

    abl_trainer = FocalLossTrainer(
        class_weights=class_weights_tensor,
        focal_gamma=2.0,
        model=abl_model,
        args=abl_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )
    abl_trainer.train()

    abl_pred   = abl_trainer.predict(val_dataset)
    abl_labels = abl_pred.label_ids
    abl_preds  = np.argmax(abl_pred.predictions, axis=1)
    f1_per     = f1_score(abl_labels, abl_preds, average=None)

    ablation2_results.append({
        'Learning Rate': lr,
        'Accuracy': accuracy_score(abl_labels, abl_preds),
        'Macro F1': f1_score(abl_labels, abl_preds, average='macro'),
        'F1-Neg':   f1_per[0],
        'F1-Pos':   f1_per[2],
        'Paper Setting': '*' if lr == 3e-5 else '',
    })
    del abl_model; torch.cuda.empty_cache()

abl2_df = pd.DataFrame(ablation2_results)
print('\n=== Ablation 2: Learning Rate Summary ===')
print(abl2_df.to_string(index=False))


# Ablation Study 3 — CrossEntropy vs Focal Loss (R6 - Reviewer Response)
**Reviewer Comment 6:** Justify loss function choice for handling class imbalance.

Comparing: (a) CrossEntropy + class weight vs (b) Focal Loss gamma=2.0 + class weight (paper setting).

**Paper Section:** Section III.B — add justification for Focal Loss selection.

**Reference:** Lin et al. (2020). *Focal Loss for Dense Object Detection*. IEEE TPAMI 42(2):318-327.

In [ ]:
# Ablation 3: CrossEntropy vs Focal Loss

class CrossEntropyTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss = F.cross_entropy(logits, labels, weight=self.class_weights)
        return (loss, outputs) if return_outputs else loss

ablation3_results = []
loss_configs = [
    ('CrossEntropy + class weight', 'CE'),
    ('Focal Loss gamma=2.0 + class weight (paper)', 'FL'),
]

for loss_name, loss_key in loss_configs:
    print(f'\n--- Training with {loss_name} ---')

    abl_model = IndoBERTClassifier(
        model_name=MODEL_NAME,
        num_labels=3,
        dropout_rate=0.1
    ).to(device)

    abl_args = TrainingArguments(
        output_dir=f"{MODEL_PATH}/abl3_{loss_key}",
        eval_strategy='epoch',
        save_strategy='no',
        learning_rate=3e-5,
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        logging_steps=200,
        report_to='none',
        fp16=True,
        seed=SEED,
    )

    if loss_key == 'CE':
        abl_trainer = CrossEntropyTrainer(
            class_weights=class_weights_tensor,
            model=abl_model,
            args=abl_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
            compute_metrics=compute_metrics,
        )
    else:
        abl_trainer = FocalLossTrainer(
            class_weights=class_weights_tensor,
            focal_gamma=2.0,
            model=abl_model,
            args=abl_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
            compute_metrics=compute_metrics,
        )

    abl_trainer.train()

    abl_pred   = abl_trainer.predict(val_dataset)
    abl_labels = abl_pred.label_ids
    abl_preds  = np.argmax(abl_pred.predictions, axis=1)
    f1_per     = f1_score(abl_labels, abl_preds, average=None)

    ablation3_results.append({
        'Loss Function': loss_name,
        'Accuracy': accuracy_score(abl_labels, abl_preds),
        'Macro F1': f1_score(abl_labels, abl_preds, average='macro'),
        'F1-Neg':   f1_per[0],
        'F1-Pos':   f1_per[2],
    })
    del abl_model; torch.cuda.empty_cache()

abl3_df = pd.DataFrame(ablation3_results)
print('\n=== Ablation 3: Loss Function Summary ===')
print(abl3_df.to_string(index=False))


# Stratified 5-Fold Cross-Validation (R10 - Reviewer Response)
**Reviewer Comment 10:** Validate robustness of reported metrics across data splits.

Running stratified 5-fold CV with `EarlyStoppingCallback(patience=3)` and `num_train_epochs=15` per fold.

**Paper Section:** Section IV — replace single train/val split results with mean +/- std from 5-fold.

**Reference:** Kohavi (1995). *A study of cross-validation and bootstrap for accuracy estimation*. IJCAI-95.

In [ ]:
# Stratified 5-Fold Cross-Validation
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
all_texts_kf  = df_seed['text'].values
all_labels_kf = df_seed['label'].map(label_to_id).values

kfold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(all_texts_kf, all_labels_kf)):
    print(f'\n{"="*40}')
    print(f'FOLD {fold+1}/5')
    print(f'{"="*40}')

    fold_train_ds = SentimentDataset(
        all_texts_kf[train_idx].tolist(),
        all_labels_kf[train_idx].tolist(),
        tokenizer, MAX_LENGTH
    )
    fold_val_ds = SentimentDataset(
        all_texts_kf[val_idx].tolist(),
        all_labels_kf[val_idx].tolist(),
        tokenizer, MAX_LENGTH
    )

    fold_model = IndoBERTClassifier(
        model_name=MODEL_NAME,
        num_labels=3,
        dropout_rate=0.1
    ).to(device)

    fold_args = TrainingArguments(
        output_dir=f"{MODEL_PATH}/kfold_fold{fold+1}",
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
        learning_rate=3e-5,
        num_train_epochs=15,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        logging_steps=200,
        report_to='none',
        fp16=True,
        seed=SEED,
    )

    fold_trainer = FocalLossTrainer(
        class_weights=class_weights_tensor,
        focal_gamma=2.0,
        model=fold_model,
        args=fold_args,
        train_dataset=fold_train_ds,
        eval_dataset=fold_val_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )
    fold_trainer.train()

    fold_pred  = fold_trainer.predict(fold_val_ds)
    fold_true  = fold_pred.label_ids
    fold_preds = np.argmax(fold_pred.predictions, axis=1)
    f1_per     = f1_score(fold_true, fold_preds, average=None)

    kfold_results.append({
        'Fold':        fold + 1,
        'Accuracy':    accuracy_score(fold_true, fold_preds),
        'Macro F1':    f1_score(fold_true, fold_preds, average='macro'),
        'Weighted F1': f1_score(fold_true, fold_preds, average='weighted'),
        'F1-Neg':  f1_per[0],
        'F1-Neu':  f1_per[1],
        'F1-Pos':  f1_per[2],
    })
    del fold_model; torch.cuda.empty_cache()

kfold_df = pd.DataFrame(kfold_results)
metric_cols = ['Accuracy', 'Macro F1', 'Weighted F1', 'F1-Neg', 'F1-Neu', 'F1-Pos']
summary_row = {'Fold': 'Mean +/- Std'}
for col in metric_cols:
    summary_row[col] = f"{kfold_df[col].mean():.4f} +/- {kfold_df[col].std():.4f}"
kfold_display = pd.concat([kfold_df, pd.DataFrame([summary_row])], ignore_index=True)
print('\n=== Stratified 5-Fold Cross-Validation Results ===')
print(kfold_display.to_string(index=False))


# Explainable AI — Integrated Gradients (R8 - Reviewer Response)
**Reviewer Comment 8:** Provide interpretability/explainability analysis.

Using `LayerIntegratedGradients` (Captum) on `bert.embeddings.word_embeddings` to identify the most influential tokens per sentiment class.
Note: IndoBERT uses `pooler_output` ([CLS] token) for global sentence-level representation.

**Paper Section:** Section IV — add XAI subsection with token attribution bar charts.

**Reference:** Sundararajan et al. (2017). *Axiomatic Attribution for Deep Networks*. ICML 2017.

In [ ]:
# XAI: Integrated Gradients (Captum)
!pip install -q captum

from captum.attr import LayerIntegratedGradients

sentiment_model.eval()

def xai_forward_func(input_ids, attention_mask):
    out = sentiment_model(input_ids=input_ids.long(), attention_mask=attention_mask)
    return out.logits

lig = LayerIntegratedGradients(
    xai_forward_func,
    sentiment_model.bert.embeddings.word_embeddings
)

label_names_xai = ['Negative', 'Neutral', 'Positive']
fig, axes = plt.subplots(1, 3, figsize=(22, 5))

for label_id_xai, label_name_xai in enumerate(label_names_xai):
    sample_text = val_df[val_df['label_id'] == label_id_xai]['text'].iloc[0]
    enc = tokenizer(
        sample_text, return_tensors='pt',
        truncation=True, max_length=MAX_LENGTH, padding=True
    )
    input_ids_xai  = enc['input_ids'].to(device)
    attn_mask_xai  = enc['attention_mask'].to(device)
    ref_ids_xai    = torch.zeros_like(input_ids_xai).to(device)

    attributions, _ = lig.attribute(
        inputs=input_ids_xai,
        baselines=ref_ids_xai,
        additional_forward_args=(attn_mask_xai,),
        target=label_id_xai,
        return_convergence_delta=True,
        n_steps=50
    )

    attr_sum = attributions.squeeze(0).sum(dim=-1).detach().cpu().numpy()
    tokens   = tokenizer.convert_ids_to_tokens(input_ids_xai.squeeze(0).cpu().numpy())

    pairs = [(t, a) for t, a in zip(tokens, attr_sum)
             if t not in ['[PAD]', '[CLS]', '[SEP]']]
    pairs_sorted = sorted(pairs, key=lambda x: abs(x[1]), reverse=True)[:10]
    top_tokens, top_attrs = zip(*pairs_sorted)

    colors = ['#d32f2f' if a > 0 else '#1565c0' for a in top_attrs]
    axes[label_id_xai].barh(range(len(top_tokens)), [abs(a) for a in top_attrs], color=colors)
    axes[label_id_xai].set_yticks(range(len(top_tokens)))
    axes[label_id_xai].set_yticklabels(top_tokens, fontsize=10)
    axes[label_id_xai].invert_yaxis()
    axes[label_id_xai].set_title(f'Top Tokens - {label_name_xai}', fontsize=12, fontweight='bold')
    axes[label_id_xai].set_xlabel('|Attribution Score|')

plt.suptitle(
    'Integrated Gradients: Top-10 Influential Tokens per Sentiment Class (IndoBERT Fine-Tuned)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()


# Computational Efficiency Analysis (R9 - Reviewer Response)
**Reviewer Comment 9:** Report model complexity and inference efficiency.

Measuring: (a) single-sample latency, (b) batch throughput, (c) parameter breakdown, (d) GPU memory.

**Paper Section:** Section IV — add efficiency subsection with latency histogram and throughput chart.

**Reference:** Strubell et al. (2019). *Energy and Policy Considerations for Deep Learning in NLP*. ACL 2019.

In [ ]:
# Computational Efficiency Analysis
import time

sentiment_model.eval()

# --- (a) Single-Sample Latency ---
enc_lat = tokenizer(
    val_df['text'].iloc[0], return_tensors='pt',
    truncation=True, max_length=MAX_LENGTH, padding=True
)
ids_lat  = enc_lat['input_ids'].to(device)
mask_lat = enc_lat['attention_mask'].to(device)

latencies = []
with torch.no_grad():
    for _ in range(5):  # warmup
        _ = sentiment_model(input_ids=ids_lat, attention_mask=mask_lat)
    for _ in range(50):
        t0 = time.perf_counter()
        _ = sentiment_model(input_ids=ids_lat, attention_mask=mask_lat)
        if device.type == 'cuda': torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

lat_arr = np.array(latencies)
print('=== Latency (ms) ===')
print(f'  Mean : {lat_arr.mean():.2f}')
print(f'  Std  : {lat_arr.std():.2f}')
print(f'  P95  : {np.percentile(lat_arr, 95):.2f}')

plt.figure(figsize=(8, 4))
plt.hist(lat_arr, bins=20, color='steelblue', edgecolor='white')
plt.xlabel('Latency (ms)')
plt.ylabel('Frequency')
plt.title('Single-Sample Inference Latency (IndoBERT Fine-Tuned)', fontweight='bold')
plt.tight_layout(); plt.show()

# --- (b) Batch Throughput ---
batch_sizes  = [1, 4, 8, 16]
throughputs  = []
sample_texts = val_df['text'].tolist()[:16]

for bs in batch_sizes:
    enc_bs = tokenizer(
        sample_texts[:bs], return_tensors='pt',
        truncation=True, max_length=MAX_LENGTH, padding=True
    )
    ids_bs  = enc_bs['input_ids'].to(device)
    mask_bs = enc_bs['attention_mask'].to(device)
    with torch.no_grad():
        for _ in range(3): _ = sentiment_model(input_ids=ids_bs, attention_mask=mask_bs)
        t0 = time.perf_counter()
        for _ in range(20): _ = sentiment_model(input_ids=ids_bs, attention_mask=mask_bs)
        if device.type == 'cuda': torch.cuda.synchronize()
        throughputs.append(bs * 20 / (time.perf_counter() - t0))

plt.figure(figsize=(7, 4))
plt.bar([str(b) for b in batch_sizes], throughputs, color='teal', edgecolor='white')
plt.xlabel('Batch Size')
plt.ylabel('Samples / sec')
plt.title('Throughput vs Batch Size (IndoBERT Fine-Tuned)', fontweight='bold')
plt.tight_layout(); plt.show()

# --- (c) Parameter Count ---
total_params     = sum(p.numel() for p in sentiment_model.parameters())
trainable_params = sum(p.numel() for p in sentiment_model.parameters() if p.requires_grad)
bert_params      = sum(p.numel() for p in sentiment_model.bert.parameters())
head_params      = sum(p.numel() for p in sentiment_model.classifier.parameters())

print('\n=== Parameter Count ===')
print(f'  Total         : {total_params:,}')
print(f'  Trainable     : {trainable_params:,}')
print(f'  IndoBERT body : {bert_params:,}')
print(f'  Classifier    : {head_params:,}')

# --- (d) GPU Memory ---
if device.type == 'cuda':
    print('\n=== GPU Memory ===')
    print(f'  Allocated : {torch.cuda.memory_allocated(device)/1e6:.1f} MB')
    print(f'  Reserved  : {torch.cuda.memory_reserved(device)/1e6:.1f} MB')
else:
    print('\n[CPU mode - GPU memory stats not available]')


# Statistical Significance Testing (R7 - Reviewer Response)
**Reviewer Comment 7:** Provide statistical significance of performance differences between models.

(a) Bootstrap CI (n=1000, seed=42) for Macro F1.
(b) McNemar's test comparing IndoBERT Fine-Tuned vs GRU.

**Paper Section:** Section IV — add significance testing subsection.

**References:** McNemar (1947), Psychometrika 12(2); Dror et al. (2018), ACL Anthology P18-1128.

In [ ]:
# Statistical Significance Testing
from scipy.stats import binom

# --- (a) Bootstrap CI for Macro F1 ---
rng = np.random.default_rng(42)
bootstrap_f1 = []
y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred)

for _ in range(1000):
    idx = rng.integers(0, len(y_true_arr), size=len(y_true_arr))
    bootstrap_f1.append(
        f1_score(y_true_arr[idx], y_pred_arr[idx], average='macro', zero_division=0)
    )

ci_low, ci_high = np.percentile(bootstrap_f1, [2.5, 97.5])
print('=== Bootstrap CI (Macro F1) - IndoBERT Fine-Tuned ===')
print(f'  Mean  : {np.mean(bootstrap_f1):.4f}')
print(f'  95% CI: [{ci_low:.4f}, {ci_high:.4f}]')

# --- (b) McNemar's Test vs GRU ---
import os
gru_preds_path = '/kaggle/working/gru_val_preds.npy'

if os.path.exists(gru_preds_path):
    gru_preds = np.load(gru_preds_path)

    if len(gru_preds) != len(y_true_arr):
        print(f'\n[WARNING] Length mismatch: GRU={len(gru_preds)}, val={len(y_true_arr)}. Skipping.')
    else:
        ft_correct  = (y_pred_arr  == y_true_arr).astype(int)
        gru_correct = (gru_preds   == y_true_arr).astype(int)

        b = int(np.sum((ft_correct == 1) & (gru_correct == 0)))
        c = int(np.sum((ft_correct == 0) & (gru_correct == 1)))

        if (b + c) > 0:
            p_mcnemar = 2 * min(
                binom.cdf(min(b, c), b + c, 0.5),
                1 - binom.cdf(min(b, c) - 1, b + c, 0.5)
            )
        else:
            p_mcnemar = 1.0

        print(f'\n=== McNemar Test: IndoBERT Fine-Tuned vs GRU ===')
        print(f'  b (FT correct, GRU wrong) : {b}')
        print(f'  c (FT wrong, GRU correct) : {c}')
        print(f'  p-value                   : {p_mcnemar:.4f}')
        if p_mcnemar < 0.05:
            print('  Result: SIGNIFICANT difference (p < 0.05)')
        else:
            print('  Result: No significant difference (p >= 0.05)')
else:
    print('\n[INFO] GRU predictions not found at /kaggle/working/gru_val_preds.npy')
    print('Run the GRU notebook first and ensure it saves:')
    print("  np.save('/kaggle/working/gru_val_preds.npy', y_pred)")
    print('Then re-run this cell.')


# Save Predictions (R7 - Required for Cross-Model Comparison)
Saving validation predictions and ground-truth labels so other notebooks can run McNemar's test against this model.

In [ ]:
# Save predictions for cross-model statistical comparison
np.save('/kaggle/working/finetuned_val_preds.npy', np.array(y_pred))
np.save('/kaggle/working/finetuned_val_true.npy',  np.array(y_true))

print('Saved: /kaggle/working/finetuned_val_preds.npy')
print('Saved: /kaggle/working/finetuned_val_true.npy')
print(f'Shape: {np.array(y_pred).shape}')


# save model

In [ ]:
final_save_path = f"{MODEL_PATH}/indobert_model"
trainer.save_model(final_save_path)

print(f"✅ Model terbaik berhasil disimpan di: {final_save_path}")